# Information flow routes: which components carry a prediction?

Every residual-stream vector is a sum of the vectors that wrote it, and each part's share of that sum can be scored from one forward pass. Walking down from the prediction and keeping only the parts above a threshold gives the routes information took to get there (Ferrando & Voita, 2024, [arXiv:2403.00824](https://arxiv.org/abs/2403.00824)). Murano's implementation is written from the paper; the authors' code, [`facebookresearch/llm-transparency-tool@f1340f0`](https://github.com/facebookresearch/llm-transparency-tool/tree/f1340f0757b959c75c139f7aa91aef16eddced67) (CC BY-NC 4.0), was used only to check test values.

**Key questions**

- Which tokens and heads does the route to an IOI prediction pass through?
- Which heads do the routes use on most prompts?
- Does a head that the routes use often also matter causally for the task?

**Structure**

1. Set up
2. Extract the routes
3. Plot head activation frequency
4. Check the heads causally
5. Save

**Model and data.** GPT-2 small in float32, loaded with `enable_attention_probs=True` because the routes need each head's attention weights. Eight indirect-object prompt pairs from `murano.tasks.ioi`; the routes use only the clean prompts.

**Requirements.** `pip install murano-interp[plot]`

## 1. Set up

In [1]:
import torch

from murano import MuranoModel, Pipeline, keys
from murano.plotting import plot_information_flow, save_figure
from murano.steps import (
    InformationFlow,
    LoadPaired,
    LogitDiffStep,
    Logits,
    Patch,
    RecoveredMetricStep,
    Save,
    SelectComponents,
)
from murano.tasks import ioi

OUTPUT_DIR = "murano_outputs/information_flow"

# GPT-2 is small enough that bfloat16 rounding degrades its predictions.
model = MuranoModel("gpt2", dtype=torch.float32, enable_attention_probs=True)
task = ioi(n=8)
print(f"{model.model_id}: {model.n_layers} layers, {model.n_heads} heads")
print("clean  :", task.clean[0], "->", repr(task.correct[0]))

gpt2: 12 layers, 12 heads
clean  : When Mary and John went to the store, John gave a drink to -> ' Mary'


## 2. Extract the routes

`InformationFlow` starts at each prompt's last token and keeps every edge whose importance exceeds `threshold`. The paper uses 0.03 for its head-frequency plots, so this notebook does too. Edge positions index the prompt's real tokens, so `result.tokens` labels them.

In [2]:
results = Pipeline([LoadPaired(task), InformationFlow(model, threshold=0.03)]).run()

flow = results[keys.INFORMATION_FLOW]
tokens, start, edges = flow.tokens[0], flow.positions[0], flow.edges[0]
print(f"prompt 0: {len(edges)} edges in the route to {tokens[start]!r}")

embedded = sorted({e.source_pos for e in edges if e.source.module == "resid_pre"})
print("tokens the route reaches at the embeddings:", [tokens[p] for p in embedded])

print("\nlayer 9+ attention edges into the last token from other tokens:")
for e in edges:
    if (
        e.dest.layer >= 9
        and e.dest.module == "resid_mid"
        and e.source_pos != e.dest_pos
    ):
        print(
            f"  L{e.dest.layer}: {tokens[e.source_pos]!r} -> {tokens[start]!r}  {e.importance:.3f}"
        )

prompt 0: 288 edges in the route to ' to'
tokens the route reaches at the embeddings: ['When', ' Mary', ' and', ' John', ' went', ' to', ' John', ' gave', ' drink', ' to']

layer 9+ attention edges into the last token from other tokens:
  L10: ' John' -> ' to'  0.044
  L9: ' Mary' -> ' to'  0.038


The late attention edges into the last token come from `' Mary'` and `' John'`, the two names. Edges are summed over heads, so the per-head view comes next.

## 3. Plot head activation frequency

`contributions` gives, for each head and MLP, the share of prompts whose routes pass through it. A head counts as used when one of its own sub-edges clears the threshold inside a route.

In [3]:
fig = plot_information_flow(flow)
save_figure(fig, f"{OUTPUT_DIR}/plots/head_frequency.png")
fig

In [4]:
heads = {n: v for n, v in flow.contributions.items() if n.module == "self_attn"}
print("most frequent heads:")
for node, value in sorted(heads.items(), key=lambda kv: -kv[1])[:6]:
    print(f"  {str(node):<20} {value:.2f}")

print("\nfrequent heads in layers 7 and up:")
late_heads = sorted(n for n, v in heads.items() if n.layer >= 7 and v >= 0.5)
for node in late_heads:
    print(f"  {str(node):<20} {heads[node]:.2f}")

most frequent heads:
  L0.self_attn.h1      1.00
  L0.self_attn.h2      1.00
  L0.self_attn.h3      1.00
  L0.self_attn.h4      1.00
  L0.self_attn.h5      1.00
  L0.self_attn.h6      1.00

frequent heads in layers 7 and up:
  L7.self_attn.h8      1.00
  L7.self_attn.h9      0.75
  L8.self_attn.h3      0.50
  L8.self_attn.h6      1.00
  L9.self_attn.h6      0.75
  L9.self_attn.h9      0.75
  L10.self_attn.h0     0.50


Layer 0 heads sit on nearly every route, because every route ends at the embeddings. The later heads include L9H9, L9H6 and L10H0, the name movers Wang et al. (2023) found in this task's circuit.

## 4. Check the heads causally

Activation frequency says a head lies on the routes. To see whether it matters for the task, each run below patches clean activations into the corrupt prompts at a set of heads and reports the recovered fraction of the clean logit difference.

In [5]:
baseline = Pipeline(
    [
        Logits(model),
        Logits(
            model,
            prompts_key=keys.CORRUPT_PROMPTS,
            logits_key=keys.CORRUPT_LOGITS,
            mask_key=keys.CORRUPT_MASK,
            targets=None,
        ),
        LogitDiffStep(task.correct, task.incorrect, output_key="ld_clean", model=model),
        LogitDiffStep(
            task.correct,
            task.incorrect,
            logits_key=keys.CORRUPT_LOGITS,
            mask_key=keys.CORRUPT_MASK,
            output_key="ld_corrupt",
            model=model,
        ),
    ]
).run(results)

recover = [
    LogitDiffStep(
        task.correct,
        task.incorrect,
        logits_key=keys.PATCHED_LOGITS,
        mask_key=keys.PATCHED_MASK,
        output_key="ld_patched",
        model=model,
    ),
    RecoveredMetricStep("ld_clean", "ld_corrupt", "ld_patched"),
]

top = Pipeline(
    [
        SelectComponents(
            keys.INFORMATION_FLOW, top_k=len(late_heads), modules="self_attn"
        ),
        Patch(model, targets_key=keys.SELECTION),
        *recover,
    ]
).run(baseline.copy())
late = Pipeline([Patch(model, targets=late_heads), *recover]).run(baseline.copy())

print("top heads by frequency:", [str(n) for n in top[keys.SELECTION].nodes])
print(f"  recovered {top[keys.RECOVERED].value:.1%}")
print("frequent heads in layers 7 and up:", [str(n) for n in late_heads])
print(f"  recovered {late[keys.RECOVERED].value:.1%}")

top heads by frequency: ['L0.self_attn.h1', 'L0.self_attn.h2', 'L0.self_attn.h3', 'L0.self_attn.h4', 'L0.self_attn.h5', 'L0.self_attn.h6', 'L0.self_attn.h7']
  recovered 12.6%
frequent heads in layers 7 and up: ['L7.self_attn.h8', 'L7.self_attn.h9', 'L8.self_attn.h3', 'L8.self_attn.h6', 'L9.self_attn.h6', 'L9.self_attn.h9', 'L10.self_attn.h0']
  recovered 49.1%


The most frequent heads are the layer 0 heads that every route crosses, and patching them recovers 12.6% of the logit difference. The same number of frequent heads from layers 7 and up, which include the name movers, recovers 49.1%.

## 5. Save

In [6]:
run_dir = Save(
    output_dir="murano_outputs", run_name="information_flow", model_id=model.model_id
)(results)[keys.OUTPUT_DIR]

print("saved to:", run_dir)

saved to: murano_outputs/information_flow


## What next

- [`activation_patching.ipynb`](activation_patching.ipynb) scores every head by patching one at a time, which takes one forward pass per head where routes take one per prompt.
- [`circuit_discovery.ipynb`](circuit_discovery.ipynb) traces how the IOI heads feed each other with path patching.